# 01 Build the dataset
CPU runtime. List the GenImage zips in place on Drive, sample a subset, extract only that and check it.

Refs: GenImage, Zhu et al. 2023 (arXiv 2306.08571)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## Run flags
Slow steps only run when their flag is on.

In [ ]:
RUN_SPEED_TEST = True    # step 4, local disk only
RUN_BENCHMARK = False    # step 4b, ~10-20 min, run once to pick the extraction settings
RUN_EXTRACTION = False   # step 11, turn on after checking step 10
RELIST = False           # redo the 7z listing even if manifests exist

In [ ]:
import json
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src import archive_utils as au
from src import image_utils as iu
from src import manifest as mf
from src.config import generator_names, get_logger

pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("01_build_dataset", P.logs)  # also saved to logs/
GEN_FOLDERS = [g["folder"] for g in cfg["generators"]]
NAMES = generator_names(cfg)
SZ = cfg["seven_zip"]
SEED = cfg["project"]["seed"]
TRAIN_GEN = cfg["train_generator"]

# Part A: access and manifest
## Step 1: archive files on Drive

In [ ]:
print(au.seven_zip_version(SZ["binary"]))

for gen in GEN_FOLDERS:
    gdir = P.genimage_root / gen
    if not gdir.exists():
        print(f"\n{gen}: FOLDER NOT FOUND at {gdir}")
        continue
    files = au.list_folder_files(gdir)
    print(f"\n{gen}: {len(files)} files, {files['gb'].sum():.1f} GB "
          f"(showing the first 3 and the last 2)")
    display(pd.concat([files.head(3), files.tail(2)]).drop_duplicates())

In [ ]:
inv = au.archive_inventory(P.genimage_root, GEN_FOLDERS)
inv.to_csv(P.manifests / "archive_inventory.csv", index=False)
display(inv)
bad = inv.loc[inv["status"] != "ok"]
if len(bad):
    log.warning(f"{len(bad)} generator(s) have a problem (see 'status'); they will be skipped later:\n{bad}")
print(f"Total archive size on Drive: {inv['total_gb'].sum():.0f} GB")

## Steps 2-3: list SD1.4 in place
Stops with the exact 7z error if the split zip can't be opened.

In [ ]:
try:
    t0 = time.time()
    m_sd, secs = mf.list_generator_to_manifest(TRAIN_GEN, inv, P.manifests, P.work_dir, cfg,
                                               relist=RELIST, logger=log)
    print(f"Listing took {time.time() - t0:.1f}s total" + (f" (7z: {secs:.1f}s)" if secs else " (loaded from disk)"))
except au.SevenZipError as e:
    raise RuntimeError(
        "STOP: 7-Zip could not open the split archive directly from Drive.\n"
        f"Exact error:\n{e}\n\n"
        "Do NOT copy the whole archive to work around this. Check that every .zNN part and the .zip "
        "are present (inventory table above) and that the Drive shortcut points at the right folder. "
        "If p7zip 16.02 cannot read this split-zip format, a newer official 7-Zip build (7zz) may: "
        "install it and set seven_zip.binary in configs/config.yaml.") from e

print(f"{len(m_sd):,} files in {TRAIN_GEN}")
display(m_sd.head())

Check the folder layout and how often a class id can be parsed from the filename.

In [ ]:
print("Top-level folders:", m_sd["top_folder"].value_counts().to_dict())
display(pd.crosstab([m_sd["split"], m_sd["label"]], m_sd["class_source"].fillna("none"), margins=True))
for (lab, src), g in m_sd.groupby(["label", m_sd["class_source"].fillna("none")]):
    print(f"{lab:<7} {src:<12} examples: {g['filename'].head(3).tolist()}")

## Step 4: extraction speed test
100 files to local disk, then real format vs extension.

In [ ]:
if RUN_SPEED_TEST:
    n = SZ["speed_test_n_per_label"]
    imgs = m_sd.loc[m_sd["is_image"] & m_sd["label"].isin(["ai", "nature"])]
    test = pd.concat([imgs.loc[imgs["label"] == lab].sample(n, random_state=SEED) for lab in ["ai", "nature"]])
    test_dir = P.work_dir / "test"
    shutil.rmtree(test_dir, ignore_errors=True)
    found, secs = [], 0.0
    for zp, g in test.groupby("archive_path"):
        res = au.extract_files(zp, g["inner_path"].tolist(), test_dir, SZ["binary"], SZ["extract_timeout_sec"])
        found += res["found"]
        secs += res["seconds"]
        if res["missing"]:
            log.warning(f"{len(res['missing'])} files missing after extraction, e.g. {res['missing'][:3]}")
    mb = test.loc[test["inner_path"].isin(found), "uncompressed_bytes"].sum() / 1e6
    print(f"Extracted {len(found)}/{len(test)} files, {mb:.1f} MB in {secs:.1f}s "
          f"-> {secs / max(len(found), 1):.2f} s/image, {mb / max(secs, 1e-9):.2f} MB/s")
    meta = pd.DataFrame([iu.image_metadata(test_dir / p, with_hashes=False) for p in found])
    meta["ext"] = [Path(p).suffix.lower() for p in found]
    meta["label"] = test.set_index("inner_path").loc[found, "label"].values
    print("\nReal format (from file content) vs extension:")
    display(pd.crosstab([meta["label"], meta["ext"]], meta["real_format"].fillna("UNREADABLE")))
    display(meta.groupby("label")[["width", "height", "file_bytes", "jpeg_quality"]].describe().T)
    est_h = secs / max(len(found), 1) * (2 * cfg["sampling"]["train_per_label"]
                                          + 2 * len(GEN_FOLDERS) * cfg["sampling"]["test_per_label"]) / 3600
    print(f"\nRough extrapolation for the full sample at this speed: ~{est_h:.1f} h "
          "(batches are larger than this test, so it is likely faster).")

## Step 4b: speed benchmark
Parallel 7z from Drive with 1/4/8 workers vs copying an archive part to local disk.

In [ ]:
if not RUN_BENCHMARK:
    print("RUN_BENCHMARK = False -> skipping the benchmark.")
else:
    bench_dir = P.work_dir / "bench"
    shutil.rmtree(bench_dir, ignore_errors=True)
    imgs = m_sd.loc[m_sd["is_image"] & m_sd["label"].isin(["ai", "nature"])]
    sd_zip = imgs["archive_path"].iloc[0]
    # different files per run so Drive caching doesn't help the later runs
    tr = imgs.loc[imgs["split"] == "train"]
    tr = tr.sample(min(600, len(tr)), random_state=SEED)
    va = imgs.loc[imgs["split"] == "val"]
    va = va.sample(min(200, len(va)), random_state=SEED)
    k = len(tr) // 3
    runs = [("train", 1, tr.iloc[:k]), ("train", 4, tr.iloc[k:2 * k]), ("train", 8, tr.iloc[2 * k:3 * k]),
            ("val", 4, va)]
    rows = []
    for split, w, files in runs:
        out = bench_dir / f"{split}_{w}"
        res = au.parallel_extract(sd_zip, files["inner_path"].tolist(), out, w, SZ["binary"], SZ["extract_timeout_sec"])
        n = len(res["found"])
        mb = files.loc[files["inner_path"].isin(res["found"]), "uncompressed_bytes"].sum() / 1e6
        rows.append({"split": split, "workers": w, "files": n, "seconds": round(res["seconds"], 1),
                     "s_per_image": res["seconds"] / max(n, 1), "MB_per_s": mb / max(res["seconds"], 1e-9)})
        if res["missing"]:
            log.warning(f"benchmark {split}/{w}: {len(res['missing'])} files missing")
        shutil.rmtree(out, ignore_errors=True)
    bench = pd.DataFrame(rows)
    display(bench.round(3))

    # copy one part to local disk
    parts = au.archive_part_files(sd_zip)[:-1]
    part = parts[len(parts) // 2] if parts else Path(sd_zip)
    copy_mbps = max(au.copy_file_timed(part, bench_dir / part.name), 1e-6)
    (bench_dir / part.name).unlink()
    print(f"Copy {part.name} ({part.stat().st_size / 1e9:.2f} GB) Drive -> local: {copy_mbps:.0f} MB/s")

    # rough estimate for the full sample
    plan_file = P.manifests / "sampling_plan.csv"
    if plan_file.exists():
        pl = mf.read_csv(plan_file)
        n_total, n_sd = len(pl), int((pl["generator"] == TRAIN_GEN).sum())
    else:
        sp = cfg["sampling"]
        n_total = 2 * sp["train_per_label"] + 2 * len(GEN_FOLDERS) * sp["test_per_label"]
        n_sd = 2 * sp["train_per_label"] + 2 * sp["test_per_label"]
    best = bench.loc[bench["split"] == "train"].sort_values("s_per_image").iloc[0]
    s_img, w_best = best["s_per_image"], int(best["workers"])
    LOCAL_SPEEDUP = 10  # guess (not measured): extracting from local disk vs Drive
    sd_gb = inv.loc[inv["generator"] == TRAIN_GEN, "total_gb"].sum()
    h_drive = n_total * s_img / 3600
    h_copy = sd_gb * 1e3 / copy_mbps / 3600
    h_local = h_copy + n_sd * s_img / LOCAL_SPEEDUP / 3600 + (n_total - n_sd) * s_img / 3600
    free_gb = shutil.disk_usage(P.work_dir).free / 1e9
    fits = sd_gb < free_gb - SZ["local_copy_margin_gb"]
    print(f"\nBest from Drive: {w_best} workers, {s_img:.3f} s/image")
    print(f"A) parallel from Drive, all {n_total:,} files: ~{h_drive:.1f} h")
    print(f"B) copy SD1.4 ({sd_gb:.0f} GB, ~{h_copy:.1f} h) + local extract of {n_sd:,} "
          f"(assumes {LOCAL_SPEEDUP}x faster) + Drive for the rest: ~{h_local:.1f} h"
          + ("" if fits else f"  [does NOT fit: {free_gb:.0f} GB free]"))
    if fits and h_local < h_drive:
        print(f"\nRecommendation: B. In configs/config.yaml set\n  extract_workers: {w_best}\n"
              f"  copy_local_generators: [{TRAIN_GEN}]")
    else:
        print(f"\nRecommendation: A. In configs/config.yaml set\n  extract_workers: {w_best}\n"
              "  copy_local_generators: []")

## Step 5: list all 8 generators
Resumable, failed generators are logged and skipped.

In [ ]:
manifests, status = {}, []
for gen in GEN_FOLDERS:
    try:
        t0 = time.time()
        m, secs = mf.list_generator_to_manifest(gen, inv, P.manifests, P.work_dir, cfg, relist=RELIST, logger=log)
        manifests[gen] = m
        status.append({"generator": gen, "files": len(m), "seconds": round(time.time() - t0, 1),
                       "status": "listed" if secs is not None else "loaded"})
    except Exception as e:
        log.error(f"{gen}: FAILED -> {e}")
        status.append({"generator": gen, "files": 0, "seconds": round(time.time() - t0, 1),
                       "status": f"ERROR: {str(e).splitlines()[0][:120]}"})
display(pd.DataFrame(status))

m_all = pd.concat(manifests.values(), ignore_index=True)
m_all.to_csv(P.manifests / "manifest_all.csv.gz", index=False)
print(f"manifest_all: {len(m_all):,} files from {m_all['generator'].nunique()} generators "
      f"-> {P.manifests / 'manifest_all.csv.gz'}")

## Step 6: manifest summary

In [ ]:
print("Files per generator x split x label:")
display(mf.manifest_summary(m_all))

print("File extension x label (all generators):")
display(pd.crosstab(m_all["ext"], m_all["label"], margins=True))

print("Top-level folder(s) per generator:")
display(m_all.groupby("generator")["top_folder"].unique().to_frame())

unknown = m_all.loc[(m_all["split"] == "unknown") | (m_all["label"] == "unknown")]
non_img = m_all.loc[~m_all["is_image"]]
print(f"Files with unknown split/label: {len(unknown):,}")
if len(unknown):
    display(unknown.groupby(["generator", "top_folder"]).size().rename("n").reset_index().head(20))
print(f"Non-image files: {len(non_img):,}")
if len(non_img):
    display(non_img[["generator", "inner_path", "uncompressed_bytes"]].head(20))

print("ImageNet class parsing per generator x label:")
display(pd.crosstab([m_all["generator"], m_all["label"]], m_all["class_source"].fillna("none")))

## Step 7: leakage check by filename
Do real filenames repeat across generators or train/val?

In [ ]:
leak = mf.filename_leakage_report(m_all)
print(json.dumps(leak["summary"], indent=2))
print("\nNumber of shared 'nature' filenames between generators (diagonal = total per generator):")
display(leak["overlap_matrix"])
with open(P.results / "filename_leakage_summary.json", "w") as f:
    json.dump(leak["summary"], f, indent=2)

# Part B: sampling and extraction
## Steps 8-9: draw the sample
Test reals whose filename is in the training sample are removed before sampling.

In [ ]:
plan = mf.build_sampling_plan(m_all, cfg, logger=log)
plan.to_csv(P.manifests / "sampling_plan.csv", index=False)
display(plan.groupby(["generator", "role", "label"]).size().unstack("label"))

overlap = set(plan.loc[plan.role == "train", "filename"]) & set(plan.loc[plan.role == "test", "filename"])
print(f"Filenames shared by train and test samples: {len(overlap)} (must be 0)")
assert not overlap

print("ImageNet class coverage of the sample:")
display(mf.class_coverage(plan))

## Step 10: size estimate
Stops if the sample is over the Drive budget.

In [ ]:
size = plan.groupby(["generator", "role", "label"])["uncompressed_bytes"].sum().div(1e9).round(3)
display(size.unstack("label"))
total_gb = mf.estimate_size_gb(plan)
budget = cfg["sampling"]["drive_budget_gb"]
print(f"Estimated raw subset size: {total_gb:.2f} GB (budget {budget} GB)")
if total_gb > budget:
    sug = mf.suggest_sample_sizes(plan, budget, cfg["sampling"]["train_per_label"], cfg["sampling"]["test_per_label"])
    raise RuntimeError(f"STOP: sample is over the Drive budget. Suggestion: {sug}. "
                       "Edit sampling.* in configs/config.yaml, push, and re-run.")
print("OK: within budget.")

## Step 11: extract and copy to Drive
Original bytes, resumable, parallel 7z (`extract_workers`), optional local archive copy (`copy_local_generators`).

In [ ]:
def on_file(local_path, row):
    """Metadata from the local copy."""
    return iu.image_metadata(local_path, with_hashes=True, hash_size=cfg["dedup"]["phash_size"])


if not RUN_EXTRACTION:
    print("RUN_EXTRACTION = False -> skipping extraction. Set it to True at the top to run this step.")
else:
    summary = []
    for gen in GEN_FOLDERS:
        sub = plan.loc[plan["generator"] == gen]
        if sub.empty:
            log.warning(f"{gen}: nothing sampled (listing failed?) -> skipped")
            continue
        t0 = time.time()
        log.info(f"=== {gen}: {len(sub)} files, {sub['uncompressed_bytes'].sum() / 1e9:.2f} GB ===")
        try:
            for zp, s in sub.groupby("archive_path"):
                src_zip = zp
                todo = ~s["raw_relpath"].map(lambda r: (P.project_root / r).exists())
                if gen in SZ["copy_local_generators"] and todo.any():
                    local_zip = au.copy_archive_local(zp, P.work_dir / "archives" / gen,
                                                      SZ["local_copy_margin_gb"], logger=log)
                    src_zip = local_zip if local_zip is not None else zp
                st, extra = au.extract_sample_to_drive(
                    s, src_zip, P.work_dir / "extract" / gen, P.project_root, batch_size=SZ["extract_batch_size"],
                    binary=SZ["binary"], timeout=SZ["extract_timeout_sec"], on_file=on_file, logger=log,
                    workers=SZ["extract_workers"])
                if str(src_zip) != str(zp):  # done with the local copy
                    shutil.rmtree(P.work_dir / "archives" / gen, ignore_errors=True)
                    log.info(f"{gen}: deleted local archive copy")
                st.to_csv(P.logs / f"extraction_status_{gen}.csv", mode="a", index=False,
                          header=not (P.logs / f"extraction_status_{gen}.csv").exists())
                if extra:
                    mfile = P.manifests / f"raw_metadata_{gen}.csv"
                    pd.DataFrame(extra).to_csv(mfile, mode="a", index=False, header=not mfile.exists())
                counts = st["status"].value_counts().to_dict()
                summary.append({"generator": gen, **counts, "minutes": round((time.time() - t0) / 60, 1)})
                log.info(f"{gen}: {counts}")
        except Exception as e:
            log.error(f"{gen}: extraction FAILED -> {e}")
            summary.append({"generator": gen, "error": str(e).splitlines()[0][:150]})
    display(pd.DataFrame(summary))

## Step 12: verify, hash, remove leakage
Test images that duplicate a training image (md5 or pHash) get dropped.

In [ ]:
plan = mf.read_csv(P.manifests / "sampling_plan.csv")
plan["exists"] = [(P.project_root / r).exists() for r in plan["raw_relpath"]]
print(f"Sampled files present on Drive: {plan['exists'].sum():,}/{len(plan):,}")
if not plan["exists"].all():
    display(plan.loc[~plan["exists"]].groupby(["generator", "role", "label"]).size().rename("missing"))
EXTRACTED = bool(plan["exists"].any())
if not EXTRACTED:
    print("nothing extracted yet, run step 11 first (steps 12-13 skipped)")

if EXTRACTED:
    meta_files = sorted(P.manifests.glob("raw_metadata_*.csv"))
    meta = pd.concat([mf.read_csv(f) for f in meta_files], ignore_index=True) if meta_files else pd.DataFrame(columns=["sample_id"])
    if "on_file_error" in meta:
        meta = meta.loc[meta["on_file_error"].isna()].drop(columns="on_file_error")
    meta = meta.drop_duplicates("sample_id", keep="last")
    need = plan.loc[plan["exists"] & ~plan["sample_id"].isin(meta["sample_id"])]
    print(f"Metadata available for {len(meta):,} files; computing it for {len(need):,} more from Drive...")
    if len(need):
        extra = iu.metadata_many([P.project_root / r for r in need["raw_relpath"]], n_threads=cfg["dedup"]["n_threads"],
                                 hash_size=cfg["dedup"]["phash_size"])
        extra = pd.DataFrame(extra).assign(sample_id=need["sample_id"].values)
        meta = pd.concat([meta, extra], ignore_index=True)
    meta.to_csv(P.manifests / "raw_metadata.csv", index=False)

    sub = plan.loc[plan["exists"]].merge(meta, on="sample_id", how="left")
    unreadable = sub.loc[sub["readable"] != True]
    print(f"Unreadable / corrupt files: {len(unreadable)}")
    if len(unreadable):
        display(unreadable[["sample_id", "error"]].head(20))

In [ ]:
if EXTRACTED:
    ok = sub.loc[sub["readable"] == True].reset_index(drop=True)
    pairs = mf.duplicate_pairs(ok, cfg["dedup"]["near_dup_hamming"])
    pairs.to_csv(P.manifests / "duplicate_pairs.csv", index=False)
    print(f"{len(pairs)} duplicate pairs (exact = same bytes; near = pHash distance <= {cfg['dedup']['near_dup_hamming']})")
    if len(pairs):
        display(pairs.groupby(["kind", "roles", "same_generator"]).size().rename("pairs").reset_index())
        display(pairs.groupby(["kind", "a_label", "b_label"]).size().rename("pairs").reset_index())

Eyeball a few pairs to sanity check the pHash threshold.

In [ ]:
if EXTRACTED:
    from src.eda_utils import plot_image_grid, savefig, set_style
    set_style()
    show = pairs.sort_values("hamming", ascending=False).head(6)
    if len(show):
        rel = ok.set_index("sample_id")["raw_relpath"]
        paths, caps = [], []
        for _, r in show.iterrows():
            for side in "ab":
                paths.append(P.project_root / rel[r[f"{side}_sample_id"]])
                caps.append(f"{r[f'{side}_generator'][:12]}/{r[f'{side}_role']}/{r[f'{side}_label']}\n{r['kind']} d={r['hamming']}")
        fig = plot_image_grid(paths, caps, ncols=4, title="Duplicate pairs (side by side)")
        savefig(fig, "01_duplicate_pairs", P.figures)

In [ ]:
if EXTRACTED:
    leaks = mf.leaking_test_ids(pairs) if len(pairs) else set()
    dropped = pd.concat([
        sub.loc[sub["sample_id"].isin(leaks)].assign(drop_reason="duplicate of a training image"),
        unreadable.assign(drop_reason="unreadable"),
        plan.loc[~plan["exists"]].assign(drop_reason="missing on Drive"),
    ], ignore_index=True)
    dropped.to_csv(P.manifests / "dropped_files.csv", index=False)
    log.info(f"Dropped {len(dropped)} files: {dropped['drop_reason'].value_counts().to_dict()}")
    display(dropped.groupby(["drop_reason", "generator", "role", "label"]).size().rename("n").reset_index())

## Step 13: save `subset_index.csv`
Notebooks 02 and 03 only read this file.

In [ ]:
if EXTRACTED:
    keep = sub.loc[(sub["readable"] == True) & ~sub["sample_id"].isin(leaks)].copy()
    keep["generator_name"] = keep["generator"].map(NAMES)
    cols = ["sample_id", "raw_relpath", "generator", "generator_name", "split", "role", "label", "y",
            "filename", "ext", "imagenet_class", "class_source", "uncompressed_bytes",
            "readable", "real_format", "width", "height", "mode", "file_bytes", "jpeg_quality", "jpeg_q_exact",
            "is_gray", "md5", "phash", "inner_path", "archive_path"]
    subset_index = keep[cols].sort_values(["generator", "role", "label", "filename"]).reset_index(drop=True)
    subset_index.to_csv(P.manifests / "subset_index.csv", index=False)
    print(f"Saved {P.manifests / 'subset_index.csv'}: {len(subset_index):,} images")
    display(subset_index.groupby(["generator", "role", "label"]).size().unstack("label"))
    du = sum(f.stat().st_size for f in P.raw_subset.rglob("*") if f.is_file()) / 1e9
    print(f"raw_subset/ uses {du:.2f} GB on Drive")

## Findings (fill in after running)
- Archive access: 7z version, did the split archives open in place? Listing time per generator:
- Actual inner layout vs README (top folder, split/label folders, unexpected files):
- Extraction speed (s/image, MB/s) and total extraction time:
- Benchmark (s/image with 1/4/8 workers, train vs val, copy MB/s) and the config we picked:
- Extensions / real formats of ai vs nature files:
- ImageNet class parsing (which pattern, coverage per generator):
- Filename leakage (shared nature filenames across generators / train-val):
- Duplicates found and files dropped:
- Final subset size (images, GB):